# Week 01: Student Exam Performance Data-Processing App

**Objective:** Build an end-to-end Python pipeline to:
1. Load a raw dataset of student exam records (`StudentsPerformance.csv`)
2. Clean and validate data using custom modular functions
3. Analyze exam scores using NumPy vectorized operations and statistics

*Note: This notebook is compatible with both local Jupyter environments and Google Colab.*

In [1]:
import csv
import os
import urllib.request
from typing import List, Dict, Any, Optional
import numpy as np

print(f"NumPy version: {np.__version__}")

NumPy version: 2.5.4


## 1. Load Raw Dataset
Locate the dataset (with automatic fallback download for Google Colab) and load the raw CSV data into a list of row dictionaries.

In [2]:
candidates = [
    "data/StudentsPerformance.csv",
    "week-01/data/StudentsPerformance.csv",
    "../data/StudentsPerformance.csv",
    "StudentsPerformance.csv"
]
DATA_PATH = None
for path in candidates:
    if os.path.exists(path):
        DATA_PATH = path
        break

if DATA_PATH is None:
    print("Dataset not found locally; downloading from GitHub repository...")
    url = "https://raw.githubusercontent.com/AbellTT/week1-demo/main/week-01/data/StudentsPerformance.csv"
    urllib.request.urlretrieve(url, "StudentsPerformance.csv")
    DATA_PATH = "StudentsPerformance.csv"

print(f"Using dataset: {DATA_PATH}")

def load_raw_data(filepath: str) -> List[Dict[str, str]]:
    with open(filepath, mode="r", encoding="utf-8") as f:
        return list(csv.DictReader(f))

raw_data = load_raw_data(DATA_PATH)
print(f"Loaded {len(raw_data)} raw records.")
print("Sample raw record:", raw_data[0])

Using dataset: data/StudentsPerformance.csv
Loaded 1000 raw records.
Sample raw record: {'gender': 'female', 'race/ethnicity': 'group B', 'parental level of education': "bachelor's degree", 'lunch': 'standard', 'test preparation course': 'none', 'math score': '72', 'reading score': '72', 'writing score': '74'}


## 2. Clean the Data Using Modular Functions
Validate score ranges ($0 \le \text{score} \le 100$), standardize strings, and cast boolean indicators.

In [3]:
def parse_score(val: Any) -> Optional[int]:
    try:
        score = int(float(str(val).strip()))
        return score if 0 <= score <= 100 else None
    except (ValueError, TypeError):
        return None

def clean_row(row: Dict[str, str]) -> Optional[Dict[str, Any]]:
    normalized = {k.strip().lower(): v.strip() for k, v in row.items()}
    
    math = parse_score(normalized.get("math score"))
    reading = parse_score(normalized.get("reading score"))
    writing = parse_score(normalized.get("writing score"))
    
    if math is None or reading is None or writing is None:
        return None
        
    return {
        "gender": normalized.get("gender", "unknown").lower(),
        "race_ethnicity": normalized.get("race/ethnicity", "unknown").title(),
        "parental_education": normalized.get("parental level of education", "unknown"),
        "lunch": normalized.get("lunch", "unknown").lower(),
        "test_prep_completed": normalized.get("test preparation course", "").lower() == "completed",
        "math_score": math,
        "reading_score": reading,
        "writing_score": writing,
    }

def clean_dataset(records: List[Dict[str, str]]) -> List[Dict[str, Any]]:
    cleaned = []
    for row in records:
        cleaned_row = clean_row(row)
        if cleaned_row is not None:
            cleaned.append(cleaned_row)
    return cleaned

In [4]:
cleaned_data = clean_dataset(raw_data)
print(f"Cleaned records: {len(cleaned_data)}")
print("Sample cleaned record:")
for key, value in cleaned_data[0].items():
    print(f"  {key}: {value} ({type(value).__name__})")

Cleaned records: 1000
Sample cleaned record:
  gender: female (str)
  race_ethnicity: Group B (str)
  parental_education: bachelor's degree (str)
  lunch: standard (str)
  test_prep_completed: False (bool)
  math_score: 72 (int)
  reading_score: 72 (int)
  writing_score: 74 (int)


## 3. Data Processing with NumPy
Convert records into a 2D NumPy array (`shape: (N, 3)`): Math (col 0), Reading (col 1), Writing (col 2).

In [5]:
scores = np.array(
    [[r["math_score"], r["reading_score"], r["writing_score"]] for r in cleaned_data],
    dtype=np.float64
)

total_scores = np.sum(scores, axis=1)
avg_scores = np.mean(scores, axis=1)

print(f"Scores shape: {scores.shape}")
print(f"First 5 student averages: {np.round(avg_scores[:5], 2)}")
print(f"Highest individual average: {np.max(avg_scores):.2f}")
print(f"Lowest individual average:  {np.min(avg_scores):.2f}")

Scores shape: (1000, 3)
First 5 student averages: [72.67 82.33 92.67 49.33 76.33]
Highest individual average: 100.00
Lowest individual average:  9.00


### 3.1 Descriptive Statistics per Subject

In [6]:
subjects = ["Math", "Reading", "Writing"]

print(f"{'Subject':<10} | {'Mean':>7} | {'Median':>7} | {'Std Dev':>7} | {'Min':>5} | {'Max':>5} | {'IQR':>7}")
print("-" * 66)

for idx, subj in enumerate(subjects):
    col = scores[:, idx]
    mean_val = np.mean(col)
    median_val = np.median(col)
    std_val = np.std(col)
    min_val = np.min(col)
    max_val = np.max(col)
    iqr_val = np.percentile(col, 75) - np.percentile(col, 25)
    print(f"{subj:<10} | {mean_val:>7.2f} | {median_val:>7.2f} | {std_val:>7.2f} | {min_val:>5.0f} | {max_val:>5.0f} | {iqr_val:>7.2f}")

Subject    |    Mean |  Median | Std Dev |   Min |   Max |     IQR
------------------------------------------------------------------
Math       |   66.09 |   66.00 |   15.16 |     0 |   100 |   20.00
Reading    |   69.17 |   70.00 |   14.59 |    17 |   100 |   20.00
Writing    |   68.05 |   69.00 |   15.19 |    10 |   100 |   21.25


### 3.2 Boolean Masking & Performance Filtering

In [7]:
passing_mask = np.all(scores >= 60.0, axis=1)
pass_count = int(np.sum(passing_mask))
pass_rate = (pass_count / len(scores)) * 100

p90_threshold = float(np.percentile(avg_scores, 90))
top_performers_mask = avg_scores >= p90_threshold
top_count = int(np.sum(top_performers_mask))

print(f"Passed all subjects (>= 60): {pass_count}/{len(scores)} ({pass_rate:.1f}%)")
print(f"Top 10% cutoff threshold:    {p90_threshold:.2f} avg score ({top_count} students)")

Passed all subjects (>= 60): 603/1000 (60.3%)
Top 10% cutoff threshold:    86.00 avg score (102 students)


### 3.3 Comparative Group Analysis: Impact of Test Preparation

In [8]:
prep_mask = np.array([r["test_prep_completed"] for r in cleaned_data], dtype=bool)

completed_mean = float(np.mean(avg_scores[prep_mask]))
none_mean = float(np.mean(avg_scores[~prep_mask]))
diff = completed_mean - none_mean

print(f"Completed Test Prep ({int(np.sum(prep_mask))} students): {completed_mean:.2f} avg")
print(f"No Test Prep        ({int(np.sum(~prep_mask))} students): {none_mean:.2f} avg")
print(f"Difference                                : +{diff:.2f} points")

Completed Test Prep (358 students): 72.67 avg
No Test Prep        (642 students): 65.04 avg
Difference                                : +7.63 points


### 3.4 Vectorized Z-Score Normalization

In [9]:
means = np.mean(scores, axis=0)
stds = np.std(scores, axis=0)
z_scores = (scores - means) / stds

print("Sample Standardized Z-Scores (First 3 students):")
for i in range(3):
    z_text = ", ".join([f"{subjects[j]}: {z_scores[i, j]:+.2f}" for j in range(3)])
    print(f"  Student {i+1} -> {z_text}")

Sample Standardized Z-Scores (First 3 students):
  Student 1 -> Math: +0.39, Reading: +0.19, Writing: +0.39
  Student 2 -> Math: +0.19, Reading: +1.43, Writing: +1.31
  Student 3 -> Math: +1.58, Reading: +1.77, Writing: +1.64


## 4. Key Findings & Conclusion
- **Overall Performance:** Reading has the highest overall average score (69.17), while Math has the lowest (66.09).
- **Pass Rate:** 60.3% of students achieved a passing grade ($\ge 60$) in all three subjects.
- **Test Preparation Course:** Students who completed the test preparation course scored on average **7.63 points higher** (72.67 vs 65.04) than those who did not, indicating a clear positive correlation.